In [7]:
!pip install Flask pandas matplotlib seaborn folium scikit-learn
!npm install -g localtunnel




⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
changed 22 packages in 1s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋

In [8]:
import pandas as pd
import folium
from folium.plugins import HeatMap
from sklearn.cluster import KMeans

# 1. Load your dataset sample (Make sure 'cleaned_chicago_crimes.csv' is in your Colab files panel)
output_file = "cleaned_chicago_crimes.csv"
df = pd.read_csv(output_file, nrows=20000).dropna(subset=['Latitude', 'Longitude'])

chicago_center = [41.8781, -87.6298]

# --- GENERATE MAP 1: HEATMAP ---
hotspot_map = folium.Map(location=chicago_center, zoom_start=11, tiles="OpenStreetMap")
heat_data = df[['Latitude', 'Longitude']].values.tolist()
HeatMap(heat_data, radius=10, max_zoom=13).add_to(hotspot_map)
hotspot_map.save("templates/heatmap.html")

# --- GENERATE MAP 2: K-MEANS CLUSTERS ---
cluster_map = folium.Map(location=chicago_center, zoom_start=11)
kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(df[['Latitude', 'Longitude']])

colors = ['red', 'blue', 'green', 'purple', 'orange']
for _, row in df.sample(1000, random_state=42).iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=3,
        color=colors[int(row['Cluster'])],
        fill=True,
        fill_opacity=0.6
    ).add_to(cluster_map)

cluster_map.save("templates/clusters.html")
print("✅ Interactive maps saved inside templates folder.")

✅ Interactive maps saved inside templates folder.


In [9]:
%%writefile templates/dashboard.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Chicago Crime Geospatial Dashboard</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body { background-color: #f3f4f6; font-family: system-ui, -apple-system, sans-serif; }
        .metric-card { border-left: 5px solid #dc3545; transition: transform 0.2s; background-color: #ffffff; }
        .metric-card:hover { transform: translateY(-3px); }

        .map-container, .chart-wrapper {
            border-radius: 12px;
            overflow: hidden;
            box-shadow: 0 4px 12px rgba(0,0,0,0.06);
            background-color: #ffffff;
            padding: 15px;
            border: 1px solid rgba(0,0,0,0.05);
        }

        /* --------------------------------------------------------------------
           ANIMATIONS & THEMES (Triggered dynamically on active tabs)
           -------------------------------------------------------------------- */

        /* Keyframe for a smooth pop-and-slide up entry effect */
        @keyframes slideUpFadeIn {
            0% { opacity: 0; transform: translateY(20px); }
            100% { opacity: 1; transform: translateY(0); }
        }

        /* Keyframe for a slight horizontal sweep effect */
        @keyframes sweepIn {
            0% { opacity: 0; transform: translateX(-15px); }
            100% { opacity: 1; transform: translateX(0); }
        }

        /* Keyframe for a clean scale/zoom entry effect */
        @keyframes scaleUpIn {
            0% { opacity: 0; transform: scale(0.97); }
            100% { opacity: 1; transform: scale(1); }
        }

        /* TAB 1: Heatmap Tab Configuration */
        .tab-pane.theme-heatmap.active {
            background-color: #fff1f2 !important; /* Soft Rose/Red tint */
            animation: slideUpFadeIn 0.5s ease-out forwards;
            border-radius: 0 0 12px 12px;
            transition: background-color 0.4s ease;
        }

        /* TAB 2: District Profile Tab Configuration */
        .tab-pane.theme-districts.active {
            background-color: #f0fdfa !important; /* Soft Teal/Light Blue-Green tint */
            animation: sweepIn 0.5s ease-out forwards;
            border-radius: 0 0 12px 12px;
            transition: background-color 0.4s ease;
        }

        /* TAB 3: K-Means Tab Configuration */
        .tab-pane.theme-kmeans.active {
            background-color: #fef8e7 !important; /* Soft Pale Cream/Yellow tint */
            animation: scaleUpIn 0.4s ease-out forwards;
            border-radius: 0 0 12px 12px;
            transition: background-color 0.4s ease;
        }

        /* Tab styling rules */
        .nav-tabs { border-bottom: none; }
        .nav-tabs .nav-link { border: 1px solid transparent; color: #4b5563; transition: all 0.2s ease; }
        .nav-tabs .nav-link.active {
            background-color: #ffffff;
            font-weight: 700;
            border-top: 3px solid #dc3545;
            color: #111827;
        }
        .chart-img { max-width: 100%; height: auto; border-radius: 8px; }
    </style>
</head>
<body>
    <div class="container-fluid py-4">
        <header class="pb-3 mb-4 border-bottom d-flex justify-content-between align-items-center">
            <div>
                <h1 class="text-dark fw-bold h3 mb-0">🚨 Chicago Crime Advanced Geospatial Analytics Portal</h1>
                <p class="text-muted small mb-0">Flask App running on Google Colab Environment</p>
            </div>
            <span class="badge bg-danger px-3 py-2">Data Stream: Active</span>
        </header>

        <div class="row g-3 mb-4">
            <div class="col-md-4">
                <div class="card metric-card shadow-sm p-3">
                    <div class="text-muted text-uppercase small fw-bold">Total Logged Incidents</div>
                    <div class="h2 fw-bold text-dark mt-1">{{ metrics.total_incidents }}</div>
                </div>
            </div>
            <div class="col-md-4">
                <div class="card metric-card shadow-sm p-3" style="border-left-color: #fd7e14;">
                    <div class="text-muted text-uppercase small fw-bold">Primary Criminal Typology</div>
                    <div class="h2 fw-bold text-dark mt-1 text-truncate" title="{{ metrics.top_crime }}">{{ metrics.top_crime }}</div>
                </div>
            </div>
            <div class="col-md-4">
                <div class="card metric-card shadow-sm p-3" style="border-left-color: #0d6efd;">
                    <div class="text-muted text-uppercase small fw-bold">Monitored Police Districts</div>
                    <div class="h2 fw-bold text-dark mt-1">{{ metrics.monitored_districts }}</div>
                </div>
            </div>
        </div>

        <div class="card shadow-sm border-0">
            <div class="card-header bg-white pt-3 border-0 pb-0">
                <ul class="nav nav-tabs card-header-tabs" id="dashboardTabs" role="tablist">
                    <li class="nav-item"><button class="nav-link active" id="heatmap-tab" data-bs-toggle="tab" data-bs-target="#heatmap-content" type="button">🔥 Crime Hotspot Heatmap</button></li>
                    <li class="nav-item"><button class="nav-link" id="district-tab" data-bs-toggle="tab" data-bs-target="#district-content" type="button">🛡️ District Hazard Profiles</button></li>
                    <li class="nav-item"><button class="nav-link" id="ml-tab" data-bs-toggle="tab" data-bs-target="#ml-content" type="button">🤖 K-Means Spatial Clusters</button></li>
                </ul>
            </div>

            <div class="card-body p-0">
                <div class="tab-content" id="dashboardTabsContent">

                    <div class="tab-pane fade show active p-4 theme-heatmap" id="heatmap-content" role="tabpanel">
                        <h4 class="fw-bold mb-1 text-danger">Kernel Density Estimation Overlay</h4>
                        <p class="text-muted small mb-4">Spatially correlates regional patterns across the city grid layer.</p>
                        <div class="map-container" style="height: 550px;">
                            {{ heatmap_html|safe }}
                        </div>
                    </div>

                    <div class="tab-pane fade p-4 theme-districts" id="district-content" role="tabpanel">
                        <h4 class="fw-bold mb-1" style="color: #0f766e;">Administrative Unit Risk Profiles</h4>
                        <p class="text-muted small mb-4">Isolating localized total incident densities across registered sector beats.</p>
                        <div class="row g-4">
                            <div class="col-lg-6">
                                <div class="chart-wrapper text-center">
                                    <h6 class="fw-bold text-muted mb-2 small">Top 10 High-Incident Frequencies</h6>
                                    <img src="data:image/png;base64,{{ bar_chart }}" class="chart-img" alt="District Crime Chart">
                                </div>
                            </div>
                            <div class="col-lg-6">
                                <div class="chart-wrapper">
                                    <h6 class="fw-bold text-muted mb-2 small">Top 5 Hazard Anchors (Centroids)</h6>
                                    <div style="height: 380px; width:100%; border-radius:6px; overflow:hidden;">
                                        {{ district_map_html|safe }}
                                    </div>
                                </div>
                            </div>
                        </div>
                    </div>

                    <div class="tab-pane fade p-4 theme-kmeans" id="ml-content" role="tabpanel">
                        <h4 class="fw-bold mb-1" style="color: #b45309;">Mathematical Spatial Grouping Matrix</h4>
                        <p class="text-muted small mb-4">Unsupervised machine learning partitions coordinate nodes into distinct mathematical clusters.</p>
                        <div class="text-center chart-wrapper mx-auto" style="max-width: 700px;">
                            <img src="data:image/png;base64,{{ kmeans_plot }}" class="chart-img" alt="K-Means Matrix">
                        </div>
                    </div>

                </div>
            </div>
        </div>
    </div>
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>

Overwriting templates/dashboard.html


In [10]:
# Creating backend application file
#This cell saves your server configuration code out to a Python file.



%%writefile app.py
import io
import base64
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
from sklearn.cluster import KMeans
from flask import Flask, render_template

app = Flask(__name__)

def fig_to_base64(fig):
    img = io.BytesIO()
    fig.savefig(img, format='png', bbox_inches='tight')
    img.seek(0)
    return base64.b64encode(img.getvalue()).decode('utf-8')

@app.route('/')
def dashboard():
    # Reading 20,000 rows to make sure Colab renders the maps smoothly
    df = pd.read_csv("cleaned_chicago_crimes.csv", nrows=20000)
    df = df.dropna(subset=['Latitude', 'Longitude', 'District'])
    df['District'] = df['District'].astype(int)

    chicago_center = [41.8781, -87.6298]

    metrics = {
        "total_incidents": f"{len(df):,}",
        "top_crime": df['Primary Type'].mode()[0],
        "monitored_districts": df['District'].nunique()
    }

    # 1. Heatmap Setup
    hotspot_map = folium.Map(location=chicago_center, zoom_start=11, tiles="OpenStreetMap")
    HeatMap(df[['Latitude', 'Longitude']].values.tolist(), radius=12, blur=8, max_zoom=13, max_intensity=0.8).add_to(hotspot_map)
    heatmap_html = hotspot_map._repr_html_()

    # 2. District Setup
    district_counts = df['District'].value_counts().reset_index()
    district_counts.columns = ['District', 'Crime_Count']
    dangerous_districts = district_counts.sort_values(by='Crime_Count', ascending=False)

    plt.figure(figsize=(10, 5))
    sns.barplot(data=dangerous_districts.head(10), x='District', y='Crime_Count', palette='Reds_r')
    plt.title("Top 10 High-Incident Crime Districts")
    bar_chart_b64 = fig_to_base64(plt.gcf())
    plt.close()

    district_map = folium.Map(location=chicago_center, zoom_start=10)
    district_locations = df.groupby('District')[['Latitude', 'Longitude']].mean().reset_index()
    district_summary = pd.merge(district_locations, dangerous_districts, on='District')
    for _, row in district_summary.sort_values(by='Crime_Count', ascending=False).head(5).iterrows():
        folium.Marker(location=[row['Latitude'], row['Longitude']], icon=folium.Icon(color='red')).add_to(district_map)
    district_map_html = district_map._repr_html_()

    # 3. K-Means Setup
    kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10)
    df['Cluster'] = kmeans.fit_predict(df[['Latitude', 'Longitude']])

    plt.figure(figsize=(7, 7))
    sns.scatterplot(data=df, x='Longitude', y='Latitude', hue='Cluster', palette='Set1', alpha=0.3, edgecolor=None, s=4)
    plt.scatter(kmeans.cluster_centers_[:, 1], kmeans.cluster_centers_[:, 0], marker='X', s=200, c='black', label='Centroids')
    plt.legend()
    kmeans_plot_b64 = fig_to_base64(plt.gcf())
    plt.close()

    return render_template(
        'dashboard.html', metrics=metrics, heatmap_html=heatmap_html,
        bar_chart=bar_chart_b64, district_map_html=district_map_html, kmeans_plot=kmeans_plot_b64
    )

if __name__ == '__main__':
    app.run(debug=False, port=5002)

Overwriting app.py


In [ ]:
import urllib
print("Your Tunnel Password is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

# Boot up server
!python app.py & npx localtunnel --port 5002

Your Tunnel Password is: 34.138.175.104
⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://plenty-buckets-hear.loca.lt
 * Serving Flask app 'app'
 * Debug mode: off
 * Running on http://127.0.0.1:5002
Press CTRL+C to quit
/content/app.py:45: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=dangerous_districts.head(10), x='District', y='Crime_Count', palette='Reds_r')
127.0.0.1 - - [04/Jun/2026 13:26:50] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [04/Jun/2026 13:26:53] "GET /favicon.ico HTTP/1.1" 404 -


In [ ]:
!fuser -k 5001/tcp

In [ ]:
!lsof -i :5001